## ИИММ: практическая работа 7

Суть работы:  
Предстоит реализовать простые задачи детекции и верификации лиц, а также протестировать работу различных методов трекинга объектов в различных задачах.

План работы:  
1. Детекция + верификация.
2. MIEM Lookalike.
3. Анализ устойчивости распознавания лиц.  
a) Оценить точность работы методов из DeepFace на тестовом видео.  
b) Оценить точность работы методов из DeepFace на аугментированных данных.
4. Антиспуфинг.


In [ ]:
import cv2
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
import requests
import keras
import faiss
import pickle
import numpy as np
import pandas as pd
import streamlit as st

from deepface import DeepFace
from datetime import datetime
from PIL import Image

### Этап 1. Детекция + верификация

In [ ]:
def download_yunet():
    model_url = "https://github.com/opencv/opencv_zoo/raw/refs/heads/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx"
    model_filename = "face_detection_yunet_2023mar.onnx"

    if not os.path.exists(model_filename):
        try:
            response = requests.get(model_url, stream=True)
            response.raise_for_status()
            with open(model_filename, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print("Загрузка завершена успешно.")
        except Exception as e:
            print(f"Ошибка при загрузке: {e}")


download_yunet()

In [ ]:
REFERENCE_IMG_PATH = "my_face.jpg"
MODEL_PATH = "face_detection_yunet_2023mar.onnx"
MODEL_NAME = "Facenet512"
THRESHOLD = 0.5

In [ ]:
detector = cv2.FaceDetectorYN.create(
    model=MODEL_PATH,
    config="",
    input_size=(320, 320),
    score_threshold=0.9,
    nms_threshold=0.3,
    top_k=50
)


cap = cv2.VideoCapture(0)

In [ ]:
print("Нажмите 'q' для выхода.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    height, width, _ = frame.shape
    detector.setInputSize((width, height))
    
    _, faces = detector.detect(frame)
    
    if faces is not None:
        for face in faces:

            coords = face[:4].astype(np.int32)
            x, y, w, h = coords
            
            x, y = max(0, x), max(0, y)
            face_img = frame[y:y+h, x:x+w]
            
            is_match = False
            try:

                result = DeepFace.verify(
                    img1_path=face_img, 
                    img2_path=REFERENCE_IMG_PATH, 
                    model_name=MODEL_NAME,
                    enforce_detection=True,
                    detector_backend='skip'
                )
                if result["verified"] and result["distance"] < THRESHOLD:
                    is_match = True
            except Exception as e:
                pass

            color = (0, 255, 0) if is_match else (0, 0, 255)
            label = "Me" if is_match else "Unknown"
            
            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            cv2.putText(frame, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    cv2.imshow("Face Recognition", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

### Этап 2. MIEM lookalike

In [ ]:
!curl -L -o ./photo_staff.rar https://drive.usercontent.google.com/download?id=1gfu5WynF5ZMDBfcFiVWLuA2S_uVyVQZL&export=download

In [ ]:
!tar -xf photo_staff.rar

In [ ]:
!curl -L -o ./photo_staff.csv https://drive.usercontent.google.com/download?id=1A6NivL1zPL6kLSLUOETaG_IWnJ-kg411&export=download

In [ ]:
def create_db(folder_path="photo_staff", csv_path="photo_staff.csv"):
   
    df = pd.read_csv(csv_path)
    
    file_to_name = dict(zip(df['filename'], df['name']))
    
    embeddings = []
    metadata = []

    print(f"Найдено записей в CSV: {len(df)}")
    
    for filename in os.listdir(folder_path):
        if filename in file_to_name:
            path = os.path.join(folder_path, filename)
            try:
                
                objs = DeepFace.represent(img_path=path, 
                                          model_name="ArcFace", 
                                          enforce_detection=True,
                                          detector_backend='retinaface')
                
                embedding = objs[0]["embedding"]
                
                embeddings.append(embedding)
                
                metadata.append(file_to_name[filename])
                
                if len(metadata) % 10 == 0:
                    print(f"Обработано {len(metadata)} лиц...")
                    
            except Exception as e:
                print(f"Пропуск {filename}: лицо не обнаружено или ошибка {e}")

    if not embeddings:
        print("Эмбеддинги не созданы.")
        return

    embeddings = np.array(embeddings).astype('float32')
    faiss.normalize_L2(embeddings)
    dimension = embeddings.shape[1]
    
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    faiss.write_index(index, "employees.index")
    with open("metadata.pkl", "wb") as f:
        pickle.dump(metadata, f)
    
    print(f"Проиндексировано лиц: {len(metadata)}")

In [ ]:
create_db()

In [ ]:
%%writefile app.py

import streamlit as st
import cv2
import numpy as np
import faiss
import pickle
import pandas as pd
from deepface import DeepFace
from PIL import Image
import os

@st.cache_resource
def load_resources():
    index = faiss.read_index("employees.index")
    with open("metadata.pkl", "rb") as f:
        metadata = pickle.load(f)
    df = pd.read_csv("photo_staff.csv")
    return index, metadata, df

st.set_page_config(page_title="MIEM Face ID", layout="centered")
st.title("Система распознавания сотрудников МИЭМ")
st.divider()

index, metadata, df_info = load_resources()

uploaded_file = st.file_uploader("Загрузите фото для поиска...", type=["jpg", "png", "jpeg"])

if uploaded_file is not None:
    img = Image.open(uploaded_file)
    st.image(img, caption='Ваше изображение', width=300)
    
    if st.button('🔍 Проверить по базе FAISS'):
        try:
            with st.spinner('Анализируем лицо...'):
                img_array = np.array(img.convert('RGB'))
                
                res = DeepFace.represent(img_array, model_name="ArcFace", enforce_detection=True, detector_backend='retinaface')
                target_embedding = np.array([res[0]["embedding"]]).astype('float32')
                faiss.normalize_L2(target_embedding)
                
                distance, index_id = index.search(target_embedding, k=1)
                
                match_idx = index_id[0][0]
                best_match_name = metadata[match_idx]
                dist_score = distance[0][0]

                THRESHOLD = 1.2
                
                if dist_score < THRESHOLD:
                    st.success(f"Лицо распознано!")
                else:
                    st.error("Совпадений не найдено (превышен порог расстояния)")
                    st.write(f"Ближайший результат: {best_match_name} (Dist: {dist_score:.4f})")
                col1, col2 = st.columns(2)
                with col1:
                    st.metric("Сотрудник", best_match_name)
                    st.write(f"Уровень сходства (L2): **{dist_score:.4f}**")
                
                with col2:
                    filename = df_info[df_info['name'] == best_match_name]['filename'].values[0]
                    photo_path = os.path.join("photo_staff", filename)
                    
                    if os.path.exists(photo_path):
                        st.image(photo_path, caption="Фото из личного дела")
                    else:
                        st.warning("Фото в локальной папке не найдено")

        except Exception as e:
            st.error(f"Ошибка: {e}")
            st.info("Убедитесь, что на фото отчетливо видно лицо.")

### Этап 3а. Оценить точность работы методов DeepFace на тестовом видео

In [9]:
VIDEO_PATH = "output.mp4"
CSV_PATH = "output.csv"
PHOTOS_DB_PATH = "task3_photos"
MODEL_NAME = "ArcFace"
THRESHOLD = 0.1

In [3]:
def create_db(db_path):
    embeddings = []
    names = []
    
    for f in os.listdir(db_path):
        if f.endswith(".pkl"): os.remove(os.path.join(db_path, f))

    for person_name in os.listdir(db_path):
        person_dir = os.path.join(db_path, person_name)
        if not os.path.isdir(person_dir): continue
        
        for img_name in os.listdir(person_dir):
            img_path = os.path.join(person_dir, img_name)
            try:
                res = DeepFace.represent(img_path=img_path, model_name=MODEL_NAME,
                                         detector_backend='retinaface',
                                         enforce_detection=True)
                embedding = np.array(res[0]["embedding"]).astype('float32')
                
                faiss.normalize_L2(embedding.reshape(1, -1))
                
                embeddings.append(embedding)
                names.append(person_name)
            except Exception as e:
                print(f"Пропуск {img_path}: лицо не найдено")

    embeddings = np.array(embeddings)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)

    faiss.write_index(index, "vector_db.index")
    with open("metadata.pkl", "wb") as f:
        pickle.dump(names, f)
    print(f"Индексировано лиц: {len(names)}")

In [ ]:
create_db(PHOTOS_DB_PATH)

In [10]:
def time_to_seconds(t_str):
    t = datetime.strptime(t_str, "%H:%M:%S")
    return t.hour * 3600 + t.minute * 60 + t.second

def clean_persons(p_str):
    if pd.isna(p_str): return []
    return [p.strip().replace('"', '') for p in p_str.split(',')]

df_labels = pd.read_csv(CSV_PATH)
df_labels['from_sec'] = df_labels['from'].apply(time_to_seconds)
df_labels['to_sec'] = df_labels['to'].apply(time_to_seconds)

df_labels['persons_list'] = df_labels['persons'].apply(clean_persons)

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)

output_path = "output_recognition.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

tp, fn = 0, 0
frame_idx = 0

index = faiss.read_index("vector_db.index")
with open("metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

In [ ]:
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    curr_sec = frame_idx / fps

    if frame_idx % 5 == 0:
        last_detections = []
        
        row = df_labels[(df_labels['from_sec'] <= curr_sec) & 
                        (df_labels['to_sec'] >= curr_sec)]

        try:
            faces = DeepFace.represent(
                img_path=frame,
                model_name=MODEL_NAME,
                detector_backend='retinaface',
                enforce_detection=False
            )

            for face in faces:
                target_emb = np.array(face["embedding"]).astype('float32').reshape(1, -1)
                faiss.normalize_L2(target_emb)

                similarities, indices = index.search(target_emb, k=1)
                cos_sim = similarities[0][0]
                match_name = metadata[indices[0][0]]

                facial_area = face["facial_area"]
                x, y, w, h = (
                    facial_area["x"],
                    facial_area["y"],
                    facial_area["w"],
                    facial_area["h"]
                )

                if cos_sim > THRESHOLD:
                    match_name = metadata[indices[0][0]]
                else:
                    match_name = "Unknown"

                last_detections.append({
                    "box": (x, y, w, h),
                    "name": match_name,
                    "cos_sim": cos_sim
                })

        except Exception as e:
            print("Ошибка при детекции:", e)

    for det in last_detections:
        x, y, w, h = det["box"]
        name = det["name"]
        cos_sim = det["cos_sim"]

        color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)

        cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)

        text = f"{name} | d={cos_sim:.2f} | thr={THRESHOLD}"
        cv2.putText(frame, text,
                    (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    color,
                    2)

    out.write(frame)

    frame_idx += 1

cap.release()
out.release()
cv2.destroyAllWindows()

### Этап 3б. Оценить точность работы методов из DeepFace на аугментированных данных

In [6]:
def apply_augmentation(image, mode):
    if mode == "rotate_45":
        (h, w) = image.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, 45, 1.0)
        return cv2.warpAffine(image, M, (w, h))
    
    elif mode == "rotate_90":
        return cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
    
    elif mode == "noise":
        gauss = np.random.normal(0, 20, image.shape) 
        noisy_image = image.astype('float32') + gauss
        noisy_image = np.clip(noisy_image, 0, 255)
        return noisy_image.astype('uint8')
    
    elif mode == "brightness_plus":
        return cv2.convertScaleAbs(image, alpha=1, beta=50)
    
    elif mode == "brightness_minus":
        return cv2.convertScaleAbs(image, alpha=1, beta=-50)
    
    elif mode == "blur":
        return cv2.GaussianBlur(image, (15, 15), 0)
    
    return image

In [ ]:
models = ["VGG-Face", "Facenet", "Facenet512", "OpenFace", "DeepFace", "DeepID", "ArcFace", "Dlib", "SFace", "GhostFaceNet"]

augmentations = {
    "Исходный датасет": "original",
    "Поворот на 45°": "rotate_45",
    "Поворот на 90°": "rotate_90",
    "Изображение с шумом": "noise",
    "Яркость +50%": "brightness_plus",
    "Яркость -50%": "brightness_minus",
    "Размытие": "blur"
}

REFERENCE_PATH = os.path.abspath("passport_photo.jpg")
DATASET_DIR = os.path.abspath("my_faces_20")

In [ ]:
results_table = pd.DataFrame(index=models, columns=augmentations.keys())

for model in models:
    print(f"Тестирование модели: {model}")
    for aug_name, aug_mode in augmentations.items():
        tp = 0
        total = 0
        
        for img_name in os.listdir(DATASET_DIR):
            img_path = os.path.join(DATASET_DIR, img_name)
            img = cv2.imread(img_path)
            if img is None: continue

            processed_img = apply_augmentation(img, aug_mode)
            
            try:
                res = DeepFace.verify(img1_path=REFERENCE_PATH, 
                                    img2_path=processed_img, 
                                    model_name=model, 
                                    enforce_detection=False,
                                    detector_backend='retinaface')
                if res["verified"]:
                    tp += 1
            except Exception as e:
                print(f"❌ Ошибка: {model} | {aug_name} | {img_name}: {e}")
            total += 1

        score = tp / total if total > 0 else 0
        results_table.loc[model, aug_name] = round(score, 2)

In [ ]:
results_table.to_csv("face_metrics_results.csv")

In [13]:
df = pd.read_csv('face_metrics_results.csv')
df

,Unnamed: 0,Исходный датасет,Поворот на 45°,Поворот на 90°,Изображение с шумом,Яркость +50%,Яркость -50%,Размытие
0,VGG-Face,0.85,0.75,0.60,0.70,0.80,0.70,0.70
1,Facenet,0.60,0.60,0.35,0.50,0.55,0.50,0.55
2,Facenet512,0.35,0.35,0.30,0.35,0.35,0.35,0.35
3,OpenFace,0.10,0.15,0.00,0.05,0.10,0.20,0.15
4,DeepFace,0.30,0.15,0.00,0.05,0.60,0.15,0.35
5,DeepID,0.10,0.15,0.00,0.10,0.30,0.10,0.10
6,ArcFace,1.00,0.95,0.85,0.85,1.00,0.85,0.90
7,Dlib,0.65,0.75,0.60,0.60,0.65,0.55,0.55
8,SFace,0.65,0.60,0.35,0.30,0.50,0.60,0.60
9,GhostFaceNet,0.85,0.85,0.70,0.60,0.85,0.80,0.75


### Этап 4. Антиспуфинг (моргание)

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

model_path = "face_landmarker.task"

BaseOptions = python.BaseOptions
FaceLandmarker = vision.FaceLandmarker
FaceLandmarkerOptions = vision.FaceLandmarkerOptions
VisionRunningMode = vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1,
)

landmarker = FaceLandmarker.create_from_options(options)

LEFT_EYE = [33, 160, 158, 133, 153, 144]

def eye_aspect_ratio(landmarks, indices, w, h):
    def dist(p1, p2):
        return np.linalg.norm(np.array(p1) - np.array(p2))

    points = [(int(landmarks[i].x * w), int(landmarks[i].y * h)) for i in indices]

    A = dist(points[1], points[5])
    B = dist(points[2], points[4])
    C = dist(points[0], points[3])

    return (A + B) / (2.0 * C)

cap = cv2.VideoCapture(0)
blink_frames = 0
blinks = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w, _ = frame.shape
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

    timestamp = int(cap.get(cv2.CAP_PROP_POS_MSEC))
    result = landmarker.detect_for_video(mp_image, timestamp)

    if result.face_landmarks:
        landmarks = result.face_landmarks[0]

        ear = eye_aspect_ratio(landmarks, LEFT_EYE, w, h)

        if ear < 0.2:
            blink_frames += 1
            if blink_frames > 3:
                blinks += 1
                print("✅ Моргание обнаружено! Счетчик:", blinks)
                blink_frames = 0
        else:
            blink_frames = 0

    cv2.imshow("Blink Detection (MediaPipe 0.10+)", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()